# Understanding bookings, flights and the supplied forecasts

**Purpose:** understand what the three datasets can support before choosing a model.
We will examine data quality, booking timing, market/season patterns, revenue,
and whether the forecast requirements are compatible.

**Inputs:** the three raw Delta tables in `workspace.ewh_dev`. No source CSV is
loaded into this notebook. All Databricks operations here are read-only.

**Outputs:** explanatory tables, charts and an analysis summary. This notebook
neither cleans the raw tables nor fits a model. Matching forecast totals later
will be a constraint check, not evidence of forecast accuracy.

The [research review](../docs/literature_review.md) documents 16 papers (14
full-text section reviews and two partial reviews). It motivates the questions
below; it does not establish that any method is already suitable for our data.

The case-study brief describes the inputs as dummy data. External evidence
helps assess plausibility; it cannot establish real-world causes of these rows.


## 1. Connect to the data

Locally, use the project Jupyter kernel and your existing Databricks CLI OAuth
profile. The setup cell checks `DATABRICKS_CLI`, then PATH, then this Jupyter
server's existing `/tmp/ewh-databricks-cli/databricks` installation. An environment
variable set in another terminal does not update an already running kernel.
The `/tmp` installation is temporary; on another machine or after it is removed,
install the CLI and set `DATABRICKS_CLI` to the new executable path.
In a Databricks notebook attached to compute, the same SELECT statements use
`spark.sql` instead. The selected serverless environment must provide compatible
pandas, NumPy and Matplotlib versions; the local Poetry environment is separate.

The three statements are stored in `sql/03_analysis_extract.sql`. We keep exact
Decimal amounts for source checks, and make separate floating-point copies for
plots and descriptive statistics. No credentials are stored in the notebook.

In [1]:
import csv
import json
import os
import shutil
import sys
import tempfile
from decimal import ROUND_HALF_UP, Decimal
from pathlib import Path

os.environ.setdefault(
    "MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "ewh-matplotlib")
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "sql/03_analysis_extract.sql").exists()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Run from this repository or its notebooks directory.")
sys.path.insert(0, str(ROOT / "src"))
from ewh_forecast.data_access import query_dataframe

PROFILE = os.getenv("DATABRICKS_CONFIG_PROFILE", "EWH_CASE_STUDY")
WAREHOUSE_ID = os.getenv("DATABRICKS_WAREHOUSE_ID", "191716ea8f6d75ad")

# Prefer explicit configuration or PATH; otherwise use this server's existing CLI.
CLI_PATH = os.getenv("DATABRICKS_CLI") or shutil.which("databricks")
if not CLI_PATH:
    server_cli = Path("/tmp/ewh-databricks-cli/databricks")
    if server_cli.is_file() and os.access(server_cli, os.X_OK):
        CLI_PATH = str(server_cli)
if "spark" not in globals():
    if not CLI_PATH:
        raise RuntimeError(
            "Databricks CLI not found. Set os.environ['DATABRICKS_CLI'] to its "
            "executable path above this cell, or install it on PATH. "
            "See notebooks/README.md."
        )
    os.environ["DATABRICKS_CLI"] = CLI_PATH
    print("Databricks CLI:", CLI_PATH)

FIGURES = ROOT / "reports/figures"
OUTPUTS = ROOT / "outputs/analysis"
FIGURES.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)
plt.rcParams.update(
    {"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False}
)
pd.set_option("display.max_rows", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

# This specific extraction file contains three SELECTs and no semicolons inside literals.
statements = [
    s.strip()
    for s in (ROOT / "sql/03_analysis_extract.sql").read_text().split(";")
    if s.strip()
]
assert len(statements) == 3
keys = ["actuals", "booking_forecast", "flight_forecast"]
raw = {}
for name, statement in zip(keys, statements):
    if "spark" in globals():
        raw[name] = globals()["spark"].sql(statement).toPandas()
    else:
        raw[name] = query_dataframe(
            statement, warehouse_id=WAREHOUSE_ID, profile=PROFILE
        )

with (ROOT / "data/sources.csv").open(newline="") as inventory:
    expected_counts = [
        int(row["row_count"]) for row in csv.DictReader(inventory) if row["row_count"]
    ]
assert [len(raw[name]) for name in keys] == expected_counts

source_totals = {
    name: {
        "rows": len(frame),
        "tix": sum(Decimal(str(value)) for value in frame.Tix),
        "revenue_eur": sum(Decimal(str(value)) for value in frame.Revenue),
    }
    for name, frame in raw.items()
}
display(pd.DataFrame(source_totals).T)

frames = {name: frame.copy() for name, frame in raw.items()}
for frame in frames.values():
    for column in frame:
        if column.endswith("Date"):
            frame[column] = pd.to_datetime(frame[column], errors="raise")
    frame["Tix"] = frame.Tix.astype(float)
    frame["Revenue"] = frame.Revenue.astype(float)
a, b, f = (frames[name] for name in keys)
a["lead_days"] = (a.FlightDate - a.BookingDate).dt.days
OBSERVATION_START = a.BookingDate.min()
OBSERVATION_END = a.BookingDate.max()
print(
    "Observed booking-date bounds:",
    OBSERVATION_START.date(),
    "to",
    OBSERVATION_END.date(),
)

Databricks CLI: /tmp/ewh-databricks-cli/databricks


,rows,tix,revenue_eur
actuals,20142,45453,33314149.00
booking_forecast,2452,23012.74,18988449.00
flight_forecast,2083,22413,18330715.00


Observed booking-date bounds: 2024-01-01 to 2025-12-31


## 2. Check the grain and basic integrity

A row is not necessarily one customer or one booking: it can contain multiple
travellers and there is no transaction identifier. The observed date/market
combinations are the keys we check. These are properties of the supplied files,
not proof that all possible transactions have been captured.

Previously, the loading script compared each typed table row with its source.
Here we recheck the analysis inputs, including calendar and market coverage.

In [2]:
quality_rows = []
coverage_rows = []
for name, frame in frames.items():
    date_cols = [column for column in frame if column.endswith("Date")]
    key_cols = [*date_cols, "FlightMarketCluster"]
    quality_rows.append(
        {
            "dataset": name,
            "rows": len(frame),
            "duplicate_keys": int(frame.duplicated(key_cols).sum()),
            "null_values": int(frame[key_cols + ["Tix", "Revenue"]].isna().sum().sum()),
            "blank_markets": int(frame.FlightMarketCluster.str.strip().eq("").sum()),
            "nonpositive_tix_rows": int(frame.Tix.le(0).sum()),
            "negative_revenue_rows": int(frame.Revenue.lt(0).sum()),
            "zero_revenue_rows": int(frame.Revenue.eq(0).sum()),
            "fractional_revenue_rows": int((frame.Revenue % 1 != 0).sum()),
            "fractional_tix_rows": int((frame.Tix % 1 != 0).sum()),
        }
    )
    for column in date_cols:
        coverage_rows.append(
            {
                "dataset": name,
                "date_column": column,
                "first_date": str(frame[column].min().date()),
                "last_date": str(frame[column].max().date()),
                "distinct_dates": frame[column].nunique(),
                "markets": frame.FlightMarketCluster.nunique(),
            }
        )
quality = pd.DataFrame(quality_rows).set_index("dataset")
display(quality)
display(pd.DataFrame(coverage_rows))
assert (
    quality[
        [
            "duplicate_keys",
            "null_values",
            "blank_markets",
            "nonpositive_tix_rows",
            "negative_revenue_rows",
        ]
    ]
    .to_numpy()
    .sum()
    == 0
)

calendar = pd.date_range("2026-01-01", "2026-12-31")
calendar_rows = []
for name, frame in [("booking_forecast", b), ("flight_forecast", f)]:
    missing_dates = calendar.difference(frame.Date.unique())
    possible = len(calendar) * frame.FlightMarketCluster.nunique()
    calendar_rows.append(
        {
            "dataset": name,
            "markets": frame.FlightMarketCluster.nunique(),
            "potential_date_market_cells": possible,
            "supplied_cells": len(frame),
            "absent_cells": possible - len(frame),
            "missing_whole_dates": ", ".join(
                d.strftime("%Y-%m-%d") for d in missing_dates
            )
            or "None",
        }
    )
display(pd.DataFrame(calendar_rows).set_index("dataset"))
print(
    "Markets present in history but absent from both forecasts:",
    sorted(
        set(a.FlightMarketCluster)
        - set(b.FlightMarketCluster)
        - set(f.FlightMarketCluster)
    ),
)
print("An absent date/market cell is unspecified. We have not replaced it with zero.")

,rows,duplicate_keys,null_values,blank_markets,nonpositive_tix_rows,negative_revenue_rows,zero_revenue_rows,fractional_revenue_rows,fractional_tix_rows
dataset,,,,,,,,,
actuals,20142,0,0,0,0,0,16,0,0
booking_forecast,2452,0,0,0,0,0,3,0,1023
flight_forecast,2083,0,0,0,0,0,2,0,0


,dataset,date_column,first_date,last_date,distinct_dates,markets
0,actuals,BookingDate,2024-01-01,2025-12-31,731,15
1,actuals,FlightDate,2024-01-04,2026-11-19,1001,15
2,booking_forecast,Date,2026-01-01,2026-12-30,364,14
3,flight_forecast,Date,2026-01-01,2026-12-31,365,14


,markets,potential_date_market_cells,supplied_cells,absent_cells,missing_whole_dates
dataset,,,,,
booking_forecast,14,5110,2452,2658,2026-12-31
flight_forecast,14,5110,2083,3027,None


Markets present in history but absent from both forecasts: ['Nordics']
An absent date/market cell is unspecified. We have not replaced it with zero.


## 3. Identify anomalies without rewriting history

A flight before its booking date cannot represent a normal booking lead time.
We report that record separately and exclude it **only from timing analysis**.
All raw records remain in the tables and in raw-total comparisons. Zero revenue
is retained: the files do not explain whether it is a genuine free product,
a correction, or a data issue. Fractional forecast tickets are expected values,
so we do not round them into whole people.

In [3]:
negative_lead = a.loc[a.lead_days.lt(0)]
display(
    pd.DataFrame(
        [
            {
                "issue": "Flight precedes booking",
                "rows": len(negative_lead),
                "tix": negative_lead.Tix.sum(),
                "revenue_eur": negative_lead.Revenue.sum(),
                "minimum_lead_days": a.lead_days.min(),
            }
        ]
    )
)
timing = a.loc[a.lead_days.ge(0)].copy()
assert len(timing) + len(negative_lead) == len(a)
assert timing.Tix.sum() + negative_lead.Tix.sum() == a.Tix.sum()
print(
    "Timing-analysis rows:",
    len(timing),
    "of",
    len(a),
    "; zero-revenue rows remain included.",
)

,issue,rows,tix,revenue_eur,minimum_lead_days
0,Flight precedes booking,1,2.00,350.00,-1


Timing-analysis rows: 20141 of 20142 ; zero-revenue rows remain included.


### 3a. Review unusual values without declaring them errors

These six screening rules reproduce the numerical flags in the supplementary
anomaly review. The AOV thresholds (below EUR 100 or above EUR 5,000), more than
20 travellers per historical row, and lead times of at least 365 days are
**exploratory choices, not company-approved validity limits**. A 365-day lead is
included in this flag but is still inside our later last-365-day cohort.

Count each row once in the combined total even if it triggers multiple rules.
Ticket and revenue shares show the size of the flagged amounts, not the size
of any confirmed error. Forecast AOVs describe date/market aggregates; historical
rows are not proven individual transactions or group bookings. Dates and market
labels identify examples; DataFrame indices are not original CSV line numbers.

All flags are diagnostic. Zero revenues, extreme AOVs, high volumes and long
positive lead times remain in the main analysis. The negative-lead exclusion
from timing analysis remains the only main-analysis row exclusion. Nordics
also remains in historical summaries; its absent forecast is unspecified.


In [4]:
REVIEW_THRESHOLDS = {
    "aov_low_eur": 100,
    "aov_high_eur": 5000,
    "historical_tix_above": 20,
    "lead_days_at_least": 365,
}
review_flags = {}
review_rule_rows = []
review_dataset_rows = []
aov_examples = []
for name, frame in frames.items():
    row_aov = frame.Revenue / frame.Tix
    flags = pd.DataFrame(index=frame.index)
    historical = name == "actuals"
    flags["negative_lead"] = frame.lead_days.lt(0) if historical else False
    flags["zero_revenue"] = frame.Revenue.eq(0)
    flags["aov_above_5000"] = row_aov.gt(REVIEW_THRESHOLDS["aov_high_eur"])
    flags["historical_positive_aov_below_100"] = (
        row_aov.gt(0) & row_aov.lt(REVIEW_THRESHOLDS["aov_low_eur"])
        if historical
        else False
    )
    flags["historical_tix_above_20"] = (
        frame.Tix.gt(REVIEW_THRESHOLDS["historical_tix_above"]) if historical else False
    )
    flags["historical_lead_at_least_365"] = (
        frame.lead_days.ge(REVIEW_THRESHOLDS["lead_days_at_least"])
        if historical
        else False
    )
    review_flags[name] = flags
    for rule, mask in flags.items():
        if not historical and (
            rule == "negative_lead" or rule.startswith("historical_")
        ):
            continue
        review_rule_rows.append(
            {
                "dataset": name,
                "rule": rule,
                "flagged_rows": int(mask.sum()),
                "tix": frame.loc[mask, "Tix"].sum(),
                "revenue_eur": frame.loc[mask, "Revenue"].sum(),
                "ticket_share_pct": frame.loc[mask, "Tix"].sum()
                / frame.Tix.sum()
                * 100,
                "revenue_share_pct": frame.loc[mask, "Revenue"].sum()
                / frame.Revenue.sum()
                * 100,
            }
        )
    any_flag = flags.any(axis=1)
    review_dataset_rows.append(
        {
            "dataset": name,
            "unique_flagged_rows": int(any_flag.sum()),
            "total_rows": len(frame),
            "ticket_share_pct": frame.loc[any_flag, "Tix"].sum()
            / frame.Tix.sum()
            * 100,
            "revenue_share_pct": frame.loc[any_flag, "Revenue"].sum()
            / frame.Revenue.sum()
            * 100,
        }
    )
    price_flag = flags.aov_above_5000 | flags.historical_positive_aov_below_100
    examples = frame.loc[price_flag].copy()
    examples["dataset"] = name
    examples["row_aov_eur"] = row_aov.loc[price_flag]
    aov_examples.append(examples)

review_rules = pd.DataFrame(review_rule_rows)
review_datasets = pd.DataFrame(review_dataset_rows)
display(review_rules.set_index(["dataset", "rule"]))
display(review_datasets.set_index("dataset"))
print("Distinct flagged rows across inputs:", review_datasets.unique_flagged_rows.sum())
print("Review flags do not establish that a record is wrong.")
display(
    pd.concat(aov_examples, ignore_index=True).reindex(
        columns=[
            "dataset",
            "BookingDate",
            "FlightDate",
            "Date",
            "FlightMarketCluster",
            "Tix",
            "Revenue",
            "row_aov_eur",
        ]
    )
)
print(
    "Blank date fields above are not applicable to that dataset, not missing source dates."
)

flagged_rows    tix  \
dataset          rule                                                     
actuals          negative_lead                                 1   2.00   
                 zero_revenue                                 16  41.00   
                 aov_above_5000                                5   7.00   
                 historical_positive_aov_below_100             3   8.00   
                 historical_tix_above_20                       5 155.00   
                 historical_lead_at_least_365                 10  25.00   
booking_forecast zero_revenue                                  3   6.00   
                 aov_above_5000                                1   2.07   
flight_forecast  zero_revenue                                  2   3.00   
                 aov_above_5000                                1   3.00   

                                                    revenue_eur  \
dataset          rule                                             
actuals          negative_lead                           350.00   
                 zero_revenue                              0.00   
                 aov_above_5000                       45,274.00   
                 historical_positive_aov_below_100       530.00   
                 historical_tix_above_20              88,596.00   
                 historical_lead_at_least_365         26,288.00   
booking_forecast zero_revenue                              0.00   
                 aov_above_5000                       19,692.00   
flight_forecast  zero_revenue                              0.00   
                 aov_above_5000                       23,033.00   

                                                    ticket_share_pct  \
dataset          rule                                                  
actuals          negative_lead                                  0.00   
                 zero_revenue                                   0.09   
                 aov_above_5000                                 0.02   
                 historical_positive_aov_below_100              0.02   
                 historical_tix_above_20                        0.34   
                 historical_lead_at_least_365                   0.06   
booking_forecast zero_revenue                                   0.03   
                 aov_above_5000                                 0.01   
flight_forecast  zero_revenue                                   0.01   
                 aov_above_5000                                 0.01   

                                                    revenue_share_pct  
dataset          rule                                                  
actuals          negative_lead                                   0.00  
                 zero_revenue                                    0.00  
                 aov_above_5000                                  0.14  
                 historical_positive_aov_below_100               0.00  
                 historical_tix_above_20                         0.27  
                 historical_lead_at_least_365                    0.08  
booking_forecast zero_revenue                                    0.00  
                 aov_above_5000                                  0.10  
flight_forecast  zero_revenue                                    0.00  
                 aov_above_5000                                  0.13

,unique_flagged_rows,total_rows,ticket_share_pct,revenue_share_pct
dataset,,,,
actuals,40,20142,0.52,0.48
booking_forecast,4,2452,0.04,0.10
flight_forecast,3,2083,0.03,0.13


Distinct flagged rows across inputs: 47
Review flags do not establish that a record is wrong.


,dataset,BookingDate,FlightDate,Date,FlightMarketCluster,Tix,Revenue,row_aov_eur
0,actuals,2024-04-23,2024-08-10,NaT,Mallorca,2.00,170.00,85.00
1,actuals,2024-08-09,2024-12-21,NaT,Canaries,1.00,"5,192.00","5,192.00"
2,actuals,2024-12-19,2025-02-09,NaT,Mallorca,2.00,100.00,50.00
3,actuals,2025-02-01,2025-07-23,NaT,Greece,1.00,"6,141.00","6,141.00"
4,actuals,2025-03-07,2025-03-22,NaT,City,4.00,260.00,65.00
5,actuals,2025-09-11,2025-11-06,NaT,North Africa,2.00,"16,788.00","8,394.00"
6,actuals,2025-12-01,2026-02-01,NaT,Canaries,2.00,"11,134.00","5,567.00"
7,actuals,2025-12-24,2026-02-08,NaT,Canaries,1.00,"6,019.00","6,019.00"
8,booking_forecast,NaT,NaT,2026-09-10,North Africa,2.07,"19,692.00","9,513.04"
9,flight_forecast,NaT,NaT,2026-11-06,North Africa,3.00,"23,033.00","7,677.67"


Blank date fields above are not applicable to that dataset, not missing source dates.


### 3b. Why do fractional tickets start in August?

Fractional Tix are expected volumes; whole-euro Revenue does not establish
whether revenue was also adjusted. First count fractional values by month.
Then test the **near-one monthly scaling hypothesis** from the supplementary
review. Its approximate factors are starting assumptions, not verified original
parameters. For each supplied ticket value `t`, infer the nearest positive
integer `n` to `t / starting_factor`.

For positive values rounded half-up to two decimals, `round(n * factor, 2) = t`
requires `(t - 0.005) / n <= factor < (t + 0.005) / n`. Intersect those intervals
across a month. If the intersection is nonempty, verify one factor inside it
against **every** row using Decimal arithmetic. Record the interval and matched
count. The midpoint is one compatible factor, not a unique recovered original.
The original rounding convention is also unknown; this tests one plausible rule.

Some adjusted values round to integers. Therefore the pattern's coverage can
exceed the count of visibly fractional rows. The result is an in-sample
construction diagnostic, not independent evidence of forecast accuracy or of
the author's intention. No supplied value is reversed or replaced.

The AOV comparison holds supplied Revenue fixed and substitutes the inferred
integer total **only as a hypothetical calculation**. It cannot establish that
revenue was unchanged during generation or that inferred integers are correct.


In [5]:
SCALING_STARTING_FACTORS = {
    8: Decimal("0.8999"),
    9: Decimal("1.0341"),
    10: Decimal("1.0121"),
    11: Decimal("1.0020"),
    12: Decimal("1.0157"),
}
monthly_scaling_rows = []
compatible_monthly_factors = {}
inferred_integer_tix = {}
raw_booking = raw["booking_forecast"]
for month, group in raw_booking.groupby(pd.to_datetime(raw_booking.Date).dt.month):
    tickets = [Decimal(str(value)) for value in group.Tix]
    revenue = sum(Decimal(str(value)) for value in group.Revenue)
    ticket_total = sum(tickets)
    fractional_count = sum(value % 1 != 0 for value in tickets)
    result = {
        "month": int(month),
        "rows": len(group),
        "fractional_tix_rows": fractional_count,
        "fractional_tix_pct": fractional_count / len(group) * 100,
        "supplied_tix": ticket_total,
        "observed_aov_eur": revenue / ticket_total,
        "starting_factor": None,
        "factor_lower_inclusive": None,
        "factor_upper_exclusive": None,
        "one_compatible_factor": None,
        "reconstructed_rows": None,
        "hypothetical_integer_tix": None,
        "hypothetical_integer_base_aov_eur": None,
    }
    if month in SCALING_STARTING_FACTORS:
        seed = SCALING_STARTING_FACTORS[month]
        integers = [
            (value / seed).to_integral_value(rounding=ROUND_HALF_UP)
            for value in tickets
        ]
        if any(value <= 0 for value in integers):
            raise ValueError(
                "Scaling hypothesis needs positive inferred integer counts."
            )
        lower = max(
            (value - Decimal("0.005")) / integer
            for value, integer in zip(tickets, integers)
        )
        upper = min(
            (value + Decimal("0.005")) / integer
            for value, integer in zip(tickets, integers)
        )
        if lower >= upper:
            raise ValueError(
                f"No common scaling factor fits the stated hypothesis for month {month}."
            )
        factor = (lower + upper) / 2
        reconstructed = [
            (integer * factor).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
            for integer in integers
        ]
        matched = sum(
            original == fitted for original, fitted in zip(tickets, reconstructed)
        )
        assert matched == len(group)
        compatible_monthly_factors[int(month)] = factor
        inferred_integer_tix.update(dict(zip(group.index, integers)))
        result.update(
            {
                "starting_factor": seed,
                "factor_lower_inclusive": lower,
                "factor_upper_exclusive": upper,
                "one_compatible_factor": factor,
                "reconstructed_rows": matched,
                "hypothetical_integer_tix": sum(integers),
                "hypothetical_integer_base_aov_eur": revenue / sum(integers),
            }
        )
    monthly_scaling_rows.append(result)
monthly_scaling = pd.DataFrame(monthly_scaling_rows)
display(monthly_scaling[["month", "rows", "fractional_tix_rows", "fractional_tix_pct"]])
display(
    monthly_scaling.loc[
        monthly_scaling.month.ge(8),
        [
            "month",
            "factor_lower_inclusive",
            "factor_upper_exclusive",
            "one_compatible_factor",
            "reconstructed_rows",
        ],
    ]
)
display(
    monthly_scaling.loc[
        monthly_scaling.month.ge(8),
        [
            "month",
            "supplied_tix",
            "hypothetical_integer_tix",
            "observed_aov_eur",
            "hypothetical_integer_base_aov_eur",
        ],
    ]
)
scaling_coverage = int(monthly_scaling.loc[monthly_scaling.month.ge(8), "rows"].sum())
print("Rows compatible with the August–December hypothesis:", scaling_coverage)
print(
    "Visibly fractional rows across the full year:",
    int(monthly_scaling.fractional_tix_rows.sum()),
)

# Replacing the Tix denominator does not explain away this extreme revenue/AOV.
north_africa_example = raw_booking.loc[
    raw_booking.Date.eq(pd.Timestamp("2026-09-10"))
    & raw_booking.FlightMarketCluster.eq("North Africa")
]
for row_index, row in north_africa_example.iterrows():
    current_aov = Decimal(str(row.Revenue)) / Decimal(str(row.Tix))
    alternative_aov = Decimal(str(row.Revenue)) / inferred_integer_tix[row_index]
    print(
        "North Africa example: supplied AOV",
        round(current_aov, 2),
        "; hypothetical integer-base AOV",
        round(alternative_aov, 2),
    )

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(monthly_scaling.month, monthly_scaling.fractional_tix_pct, color="#247BA0")
ax.set(
    xticks=range(1, 13),
    ylim=(0, 105),
    xlabel="Booking month in 2026",
    ylabel="Rows with fractional Tix (%)",
    title="Fractional ticket forecasts begin in August",
)
fig.tight_layout()
fig.savefig(FIGURES / "fractional_tix_by_month.png", dpi=150)
plt.close(fig)
display(
    Markdown(
        "![Fractional Tix by month](../reports/figures/fractional_tix_by_month.png)"
    )
)

,month,rows,fractional_tix_rows,fractional_tix_pct
0,1,232,0,0.00
1,2,212,0,0.00
2,3,209,0,0.00
3,4,179,0,0.00
4,5,187,0,0.00
5,6,181,0,0.00
6,7,207,0,0.00
7,8,299,292,97.66
8,9,192,192,100.00
9,10,189,189,100.00


,month,factor_lower_inclusive,factor_upper_exclusive,one_compatible_factor,reconstructed_rows
7,8,0.899875,0.8998780487804878048780487805,0.899876524390243902439024390,299.00
8,9,1.0340625,1.034078947368421052631578947,1.034070723684210526315789474,192.00
9,10,1.012083333333333333333333333,1.012096774193548387096774194,1.012090053763440860215053764,189.00
10,11,1.00203125,1.002045454545454545454545455,1.002038352272727272727272728,179.00
11,12,1.015657894736842105263157895,1.015666666666666666666666667,1.015662280701754385964912281,186.00


,month,supplied_tix,hypothetical_integer_tix,observed_aov_eur,hypothetical_integer_base_aov_eur
7,8,2202.19,2447,799.5913159173368328799967305,719.5962402942378422558234573
8,9,1912.11,1849,818.6893013477258107535654330,846.6327744726879394267171444
9,10,2261.14,2234,838.8264326843981354537976419,849.0170098478066248880931065
10,11,1973.11,1969,847.8391980173431790422226840,849.6089385474860335195530726
11,12,2010.19,1979,928.9718882294708460394291087,943.6129358261748357756442648


Rows compatible with the August–December hypothesis: 1045
Visibly fractional rows across the full year: 1023
North Africa example: supplied AOV 9513.04 ; hypothetical integer-base AOV 9846.00


![Fractional Tix by month](../reports/figures/fractional_tix_by_month.png)

## 4. Understand the size of each destination market

**Question:** do we have enough evidence to learn a separate pattern for every
market? Count both source rows and travellers. AOV below is always total
revenue divided by total tickets, never the simple mean of row-level AOVs.

In [6]:
market = a.groupby("FlightMarketCluster").agg(
    rows=("Tix", "size"), tix=("Tix", "sum"), revenue_eur=("Revenue", "sum")
)
market["ticket_share_pct"] = market.tix / market.tix.sum() * 100
market["aov_eur_per_traveller"] = market.revenue_eur / market.tix
market = market.sort_values("tix", ascending=False)
display(market)
fig, ax = plt.subplots(figsize=(10, 5))
market.tix.sort_values().plot.barh(ax=ax, color="#247BA0")
ax.set(
    title="Historical observed ticket volume by market",
    xlabel="Travellers booked in 2024–2025",
    ylabel="",
)
fig.tight_layout()
fig.savefig(FIGURES / "market_volume.png", dpi=150)
plt.close(fig)
display(Markdown("![Market volume](../reports/figures/market_volume.png)"))

,rows,tix,revenue_eur,ticket_share_pct,aov_eur_per_traveller
FlightMarketCluster,,,,,
City,10327,"23,364.00","12,628,255.00",51.40,540.50
Mallorca,4445,"10,027.00","8,098,328.00",22.06,807.65
Greece,1668,"3,901.00","3,904,206.00",8.58,"1,000.82"
Canaries,978,"2,031.00","2,368,358.00",4.47,"1,166.10"
Portugal Beach,512,"1,072.00","1,120,155.00",2.36,"1,044.92"
Iberian Peninsula,388,858.00,"796,807.00",1.89,928.68
North Africa,356,854.00,"923,849.00",1.88,"1,081.79"
Italy Beach,365,780.00,"698,002.00",1.72,894.87
Turkey,272,731.00,"774,717.00",1.61,"1,059.80"


![Market volume](../reports/figures/market_volume.png)

## 5. Separate booking seasonality from flight seasonality

A January booking can be for a summer flight. Plotting the two date columns
separately prevents us from confusing buying behaviour with travel behaviour.
The historical file is bounded by **booking dates**: the flight-year curves
below are observed slices, not complete annual flight demand. We do not claim
year-over-year growth in flight demand from them. [Research: P03, P07](../docs/literature_review.md)

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for column, ax, title in [
    ("BookingDate", axes[0], "Booking month"),
    ("FlightDate", axes[1], "Flight month (partial cohorts)"),
]:
    data = a.assign(year=a[column].dt.year, month=a[column].dt.month)
    monthly = data.groupby(["year", "month"]).Tix.sum().unstack(0).reindex(range(1, 13))
    monthly.plot(ax=ax, marker="o")
    ax.set(
        title=title, xlabel="Month", ylabel="Observed travellers", xticks=range(1, 13)
    )
fig.suptitle("Two calendar views of bookings captured in 2024–2025")
fig.tight_layout()
fig.savefig(FIGURES / "calendar_views.png", dpi=150)
plt.close(fig)
display(Markdown("![Calendar views](../reports/figures/calendar_views.png)"))
booking_year = a.groupby(a.BookingDate.dt.year).agg(
    rows=("Tix", "size"), tix=("Tix", "sum"), revenue_eur=("Revenue", "sum")
)
booking_year["aov_eur_per_traveller"] = booking_year.revenue_eur / booking_year.tix
display(booking_year)

![Calendar views](../reports/figures/calendar_views.png)

,rows,tix,revenue_eur,aov_eur_per_traveller
BookingDate,,,,
2024,10543,"24,101.00","17,169,651.00",712.40
2025,9599,"21,352.00","16,144,498.00",756.11


## 6. How far ahead do travellers book?

Lead time means days between booking and departure. A two-traveller row carries
twice the ticket weight of a one-traveller row. The weighted median is the day
at which at least half of represented travellers have a lead time at or below
that value. These are descriptive distributions of recorded tickets, not
independent observations for a significance test. [Research: P03](../docs/literature_review.md)

The unweighted row median and traveller-weighted median answer different
questions. Below we also compare hypothetical omissions of review candidates
on the same nonnegative-lead population. This is a sensitivity check; it does
not replace any main summary, estimate a causal effect, or validate the chosen
thresholds. Market/season and per-night price checks would need adequate peer
samples and additional business fields, particularly holiday duration.


In [8]:
def weighted_quantiles(values, weights, probabilities=(0.1, 0.5, 0.9)):
    values, weights = np.asarray(values), np.asarray(weights, dtype=float)
    if len(values) == 0 or len(values) != len(weights) or np.any(weights <= 0):
        raise ValueError("Need nonempty values with matching positive weights.")
    order = np.argsort(values)
    cumulative = np.cumsum(weights[order])
    indices = np.searchsorted(
        cumulative, np.asarray(probabilities) * cumulative[-1], side="left"
    )
    return values[order][np.minimum(indices, len(values) - 1)]


# A high-volume row must dominate the traveller-weighted median.
assert weighted_quantiles([1, 10], [9, 1], [0.5])[0] == 1


def lead_summary(group):
    q10, q50, q90 = weighted_quantiles(group.lead_days, group.Tix)
    return {
        "rows": len(group),
        "tix": group.Tix.sum(),
        "mean_days": np.average(group.lead_days, weights=group.Tix),
        "p10_days": int(q10),
        "median_days": int(q50),
        "p90_days": int(q90),
        "within_30_days_pct": group.loc[group.lead_days.le(30), "Tix"].sum()
        / group.Tix.sum()
        * 100,
        "over_180_days_pct": group.loc[group.lead_days.gt(180), "Tix"].sum()
        / group.Tix.sum()
        * 100,
    }


overall_lead = lead_summary(timing)
display(pd.DataFrame([overall_lead], index=["All recorded nonnegative lead times"]))
print("Unweighted row median, for comparison:", timing.lead_days.median(), "days")
lead_market = pd.DataFrame(
    {name: lead_summary(group) for name, group in timing.groupby("FlightMarketCluster")}
).T
lead_market.index.name = "FlightMarketCluster"
display(lead_market.sort_values("median_days"))

mass = timing.groupby("lead_days").Tix.sum().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(
    timing.lead_days, weights=timing.Tix, bins=np.arange(0, 466, 15), color="#247BA0"
)
axes[0].set(
    title="Booking lead-time distribution",
    xlabel="Days before departure (15-day bins)",
    ylabel="Observed travellers",
)
axes[1].plot(mass.index, mass.cumsum() / mass.sum() * 100, color="#D36B38")
axes[1].set(
    title="Share booking within a given lead time",
    xlabel="Lead time at most this many days",
    ylabel="Share of observed travellers (%)",
    ylim=(0, 100),
)
fig.tight_layout()
fig.savefig(FIGURES / "lead_time_distribution.png", dpi=150)
plt.close(fig)
display(
    Markdown("![Lead time distribution](../reports/figures/lead_time_distribution.png)")
)

# These alternatives measure sensitivity only; `a` and `timing` remain unchanged.
review_sensitivity_rows = []
timing_flags = review_flags["actuals"].reindex(timing.index)
subsets = {"Main timing population: retain review candidates": timing}
for rule in timing_flags.columns:
    if rule != "negative_lead":
        subsets[f"Diagnostic only: omit {rule}"] = timing.loc[~timing_flags[rule]]
subsets["Diagnostic only: omit any review flag"] = timing.loc[~timing_flags.any(axis=1)]
for label, subset in subsets.items():
    review_sensitivity_rows.append(
        {
            "population": label,
            "rows": len(subset),
            "tix": subset.Tix.sum(),
            "aov_eur_per_traveller": subset.Revenue.sum() / subset.Tix.sum()
            if len(subset)
            else np.nan,
            "weighted_median_lead_days": lead_summary(subset)["median_days"]
            if len(subset)
            else np.nan,
        }
    )
review_sensitivity = pd.DataFrame(review_sensitivity_rows)
display(review_sensitivity.set_index("population"))
print(
    "These hypothetical omissions quantify sensitivity, not corrections or a recommended training population. "
    "A small overall change can hide a large effect in a small market."
)

,rows,tix,mean_days,p10_days,median_days,p90_days,within_30_days_pct,over_180_days_pct
All recorded nonnegative lead times,20141,"45,451.00",76.80,7,47,195,37.72,12.12


Unweighted row median, for comparison: 44.0 days


,rows,tix,mean_days,p10_days,median_days,p90_days,within_30_days_pct,over_180_days_pct
FlightMarketCluster,,,,,,,,
Balkans,188.00,409.00,59.55,6.00,32.00,162.00,48.17,6.85
Mallorca,"4,445.00","10,027.00",77.24,7.00,43.00,205.00,40.28,13.56
City,"10,326.00","23,362.00",72.70,8.00,45.00,185.00,39.05,10.53
Canaries,978.00,"2,031.00",74.50,8.00,47.00,181.00,35.60,10.09
Cyprus,82.00,197.00,81.54,5.00,48.00,190.00,36.04,15.23
Italy Beach,365.00,780.00,80.40,11.00,53.00,208.00,32.31,13.33
Turkey,272.00,731.00,85.91,5.00,53.00,216.00,36.25,16.14
North Africa,356.00,854.00,70.47,8.00,54.00,157.00,33.72,7.85
Balearic Islands,279.00,591.00,82.47,7.00,59.00,195.00,33.50,13.54


![Lead time distribution](../reports/figures/lead_time_distribution.png)

,rows,tix,aov_eur_per_traveller,weighted_median_lead_days
population,,,,
Main timing population: retain review candidates,20141,"45,451.00",732.96,47
Diagnostic only: omit zero_revenue,20125,"45,410.00",733.62,47
Diagnostic only: omit aov_above_5000,20136,"45,444.00",732.08,47
Diagnostic only: omit historical_positive_aov_below_100,20138,"45,443.00",733.08,47
Diagnostic only: omit historical_tix_above_20,20136,"45,296.00",733.51,47
Diagnostic only: omit historical_lead_at_least_365,20131,"45,426.00",732.79,47
Diagnostic only: omit any review flag,20102,"45,215.00",733.23,47


These hypothetical omissions quantify sensitivity, not corrections or a recommended training population. A small overall change can hide a large effect in a small market.


### 6a. Are the flagged prices unusual among similar records?

The brief explicitly describes a **dummy dataset**. We can identify unusual
relationships and their consequences, but cannot infer real commercial causes
or the synthetic-data generator from these five columns.

Compare each previously flagged AOV with other rows in the **same dataset,
market and calendar year/quarter**, excluding the candidate itself. Actuals
use flight quarter; each forecast uses its own date dimension. If fewer than
20 other rows exist, broaden to that market within the same dataset; report
the fallback and sample size. Twenty rows is a screening convention, not a
guarantee of statistical precision. The peer median is weighted by Tix.

This is a descriptive comparison of already flagged candidates, not a new
trained detector, significance test or unbiased out-of-sample score. Forecast
aggregates and historical rows stay in separate peer pools. Historical future
flight quarters remain partially observed. Duration, hotel quality and booking
status are unavailable, so a large peer ratio does not prove an error.
[Contextual anomalies: P15](../docs/literature_review.md#p15-chandola-banerjee-and-kumar-2009)


In [9]:
PEER_MIN_ROWS = 20
contextual_rows = []
for name, frame in frames.items():
    date_column = "FlightDate" if name == "actuals" else "Date"
    peers_source = frame.assign(
        row_aov=frame.Revenue / frame.Tix,
        calendar_quarter=frame[date_column].dt.to_period("Q").astype(str),
    )
    flags = review_flags[name]
    price_flags = flags.aov_above_5000 | flags.historical_positive_aov_below_100
    for row_index, row in peers_source.loc[price_flags].iterrows():
        market_peers = peers_source.loc[
            peers_source.FlightMarketCluster.eq(row.FlightMarketCluster)
        ].drop(index=row_index)
        quarter_peers = market_peers.loc[
            market_peers.calendar_quarter.eq(row.calendar_quarter)
        ]
        same_quarter = len(quarter_peers) >= PEER_MIN_ROWS
        peers = quarter_peers if same_quarter else market_peers
        median_aov = (
            weighted_quantiles(peers.row_aov, peers.Tix, [0.5])[0]
            if len(peers)
            else np.nan
        )
        contextual_rows.append(
            {
                "dataset": name,
                "date_basis": date_column,
                "date": str(row[date_column].date()),
                "market": row.FlightMarketCluster,
                "row_aov_eur": row.row_aov,
                "peer_scope": "same market and quarter"
                if same_quarter
                else "same market, all dates",
                "peer_rows": len(peers),
                "peer_tix": peers.Tix.sum(),
                "peer_weighted_median_aov_eur": median_aov,
                "aov_to_peer_median": row.row_aov / median_aov
                if median_aov > 0
                else np.nan,
            }
        )
contextual_aov = pd.DataFrame(contextual_rows)
display(contextual_aov)

market_review_rows = []
for name, group in a.groupby("FlightMarketCluster"):
    mask = review_flags["actuals"].loc[group.index].any(axis=1)
    diagnostic_subset = group.loc[~mask]
    full_aov = group.Revenue.sum() / group.Tix.sum()
    diagnostic_aov = (
        diagnostic_subset.Revenue.sum() / diagnostic_subset.Tix.sum()
        if len(diagnostic_subset)
        else np.nan
    )
    market_review_rows.append(
        {
            "market": name,
            "flagged_rows": int(mask.sum()),
            "flagged_ticket_share_pct": group.loc[mask, "Tix"].sum()
            / group.Tix.sum()
            * 100,
            "full_aov_eur": full_aov,
            "diagnostic_aov_without_flags_eur": diagnostic_aov,
            "diagnostic_aov_change_pct": (diagnostic_aov / full_aov - 1) * 100,
        }
    )
market_review_impact = pd.DataFrame(market_review_rows)
display(
    market_review_impact.loc[market_review_impact.flagged_rows.gt(0)].sort_values(
        "diagnostic_aov_change_pct"
    )
)
print("Market-level hypothetical omissions above do not alter the main analysis.")

,dataset,date_basis,date,market,row_aov_eur,peer_scope,peer_rows,peer_tix,peer_weighted_median_aov_eur,aov_to_peer_median
0,actuals,FlightDate,2024-08-10,Mallorca,85.00,same market and quarter,561,"1,276.00",823.00,0.10
1,actuals,FlightDate,2024-12-21,Canaries,"5,192.00",same market and quarter,161,314.00,"1,043.00",4.98
2,actuals,FlightDate,2025-02-09,Mallorca,50.00,same market and quarter,424,905.00,623.00,0.08
3,actuals,FlightDate,2025-07-23,Greece,"6,141.00",same market and quarter,303,719.00,"1,041.00",5.90
4,actuals,FlightDate,2025-03-22,City,65.00,same market and quarter,1133,"2,415.00",403.00,0.16
5,actuals,FlightDate,2025-11-06,North Africa,"8,394.00",same market and quarter,57,126.00,"1,000.67",8.39
6,actuals,FlightDate,2026-02-01,Canaries,"5,567.00",same market and quarter,63,133.00,"1,244.00",4.48
7,actuals,FlightDate,2026-02-08,Canaries,"6,019.00",same market and quarter,63,134.00,"1,244.00",4.84
8,booking_forecast,Date,2026-09-10,North Africa,"9,513.04",same market and quarter,48,140.50,"1,135.81",8.38
9,flight_forecast,Date,2026-11-06,North Africa,"7,677.67",same market and quarter,38,141.00,"1,051.83",7.30


,market,flagged_rows,flagged_ticket_share_pct,full_aov_eur,diagnostic_aov_without_flags_eur,diagnostic_aov_change_pct
12,North Africa,1,0.23,"1,081.79","1,064.63",-1.59
2,Canaries,3,0.20,"1,166.10","1,157.38",-0.75
5,Greece,4,0.21,"1,000.82",999.50,-0.13
14,Turkey,2,1.09,"1,059.80","1,059.92",0.01
3,City,12,0.50,540.50,540.64,0.03
6,Iberian Peninsula,1,0.23,928.68,930.85,0.23
9,Mallorca,17,0.97,807.65,811.48,0.47


Market-level hypothetical omissions above do not alter the main analysis.


### 6b. Zero revenue, long horizons and calendar gaps

Group historical zero revenues by market and booking year to see their spread;
this cannot distinguish free products, adjustments, missing prices or generated
test cases. None of those explanations is confirmed. Check what activity is
actually recorded on 31 December, without imputing the absent forecast date.

Keep long positive lead times. Eurowings announced on 13 June 2024 that flights
through 25 October 2025 would be bookable from 17 June 2024: an advance horizon
of 495 days at the end of that schedule. This supports the plausibility of
more-than-one-year flight booking, not the validity or package availability of
any individual dummy record.
[Official announcement](https://newscloudbusiness.eurowings.com/en/ready-for-summer-2025/).

**Forecast implications matter more than arbitrary cutoffs.** With nonnegative
revenue, a zero-revenue margin forces every contributing revenue allocation to
zero. If every positive-ticket cell were required to have AOV between EUR 100
and EUR 5,000 (and zero-ticket cells carried no revenue), its marginal AOV would
also fall inside those bounds. The table below identifies supplied targets
outside that illustrative interval. The review thresholds therefore cannot
become hard model constraints while preserving all targets. This necessary
condition is not a complete joint-feasibility test.


In [10]:
zero_revenue_context = (
    a.loc[a.Revenue.eq(0)]
    .assign(booking_year=lambda x: x.BookingDate.dt.year)
    .groupby(["FlightMarketCluster", "booking_year"])
    .agg(
        rows=("Tix", "size"),
        tix=("Tix", "sum"),
        first_booking=("BookingDate", "min"),
        last_booking=("BookingDate", "max"),
    )
)
display(zero_revenue_context)

year_end_rows = []
for name, frame in frames.items():
    date_column = "BookingDate" if name == "actuals" else "Date"
    for year in sorted(frame[date_column].dt.year.unique()):
        selected = frame.loc[
            frame[date_column].eq(pd.Timestamp(year=int(year), month=12, day=31))
        ]
        year_end_rows.append(
            {
                "dataset": name,
                "date_basis": date_column,
                "date": f"{year}-12-31",
                "supplied_rows": len(selected),
                "observed_tix": selected.Tix.sum() if len(selected) else np.nan,
            }
        )
year_end_coverage = pd.DataFrame(year_end_rows)
display(year_end_coverage)
print("No supplied rows means unspecified, not a confirmed zero-activity day.")

long_lead_records = a.loc[a.lead_days.ge(REVIEW_THRESHOLDS["lead_days_at_least"])]
display(long_lead_records)
print("Observed long-lead travellers:", long_lead_records.Tix.sum())

bound_rows = []
for name, frame in [("booking_forecast", b), ("flight_forecast", f)]:
    aov = frame.Revenue / frame.Tix
    outside = aov.lt(REVIEW_THRESHOLDS["aov_low_eur"]) | aov.gt(
        REVIEW_THRESHOLDS["aov_high_eur"]
    )
    selected = frame.loc[outside].assign(
        dataset=name, marginal_aov_eur=aov.loc[outside]
    )
    bound_rows.append(selected)
forecast_aov_bound_conflicts = pd.concat(bound_rows, ignore_index=True)
display(
    forecast_aov_bound_conflicts[
        ["dataset", "Date", "FlightMarketCluster", "Tix", "Revenue", "marginal_aov_eur"]
    ]
)
print(
    "Targets outside the illustrative AOV interval:", len(forecast_aov_bound_conflicts)
)

rows   tix first_booking last_booking
FlightMarketCluster booking_year                                       
City                2024             4  8.00    2024-05-16   2024-11-14
                    2025             2  8.00    2025-01-16   2025-09-29
Greece              2024             1  2.00    2024-05-02   2024-05-02
Iberian Peninsula   2024             1  2.00    2024-02-08   2024-02-08
Mallorca            2024             5 12.00    2024-02-19   2024-12-13
                    2025             3  9.00    2025-03-06   2025-05-30

,dataset,date_basis,date,supplied_rows,observed_tix
0,actuals,BookingDate,2024-12-31,25,57.00
1,actuals,BookingDate,2025-12-31,9,18.00
2,booking_forecast,Date,2026-12-31,0,NaN
3,flight_forecast,Date,2026-12-31,4,18.00


No supplied rows means unspecified, not a confirmed zero-activity day.


,BookingDate,FlightDate,FlightMarketCluster,Tix,Revenue,lead_days
6016,2024-07-12,2025-09-29,Mallorca,2.00,"1,987.00",444
6159,2024-07-17,2025-07-25,Greece,3.00,"5,324.00",373
6374,2024-07-27,2025-10-18,Turkey,4.00,"3,558.00",448
6440,2024-07-29,2025-09-20,Greece,2.00,"1,696.00",418
6563,2024-08-03,2025-08-22,Mallorca,4.00,"3,667.00",384
6748,2024-08-10,2025-10-20,Turkey,4.00,"4,834.00",436
16888,2025-08-02,2026-08-04,City,2.00,"1,304.00",367
18144,2025-09-18,2026-09-24,Mallorca,1.00,"1,490.00",371
18295,2025-09-24,2026-10-02,Mallorca,1.00,710.00,373
18540,2025-10-08,2026-10-08,Mallorca,2.00,"1,718.00",365


Observed long-lead travellers: 25.0


,dataset,Date,FlightMarketCluster,Tix,Revenue,marginal_aov_eur
0,booking_forecast,2026-02-04,Iberian Peninsula,2.00,0.00,0.00
1,booking_forecast,2026-02-11,Greece,2.00,0.00,0.00
2,booking_forecast,2026-04-02,Mallorca,2.00,0.00,0.00
3,booking_forecast,2026-09-10,North Africa,2.07,"19,692.00","9,513.04"
4,flight_forecast,2026-05-29,Canaries,2.00,0.00,0.00
5,flight_forecast,2026-11-06,North Africa,3.00,"23,033.00","7,677.67"
6,flight_forecast,2026-11-23,Balkans,1.00,0.00,0.00


Targets outside the illustrative AOV interval: 7


### 6c. Are zero revenues connected across the three inputs?

Compare zero-revenue market coverage and ticket-count frequencies. Test two
explicit historical alignment rules within the **same market and date dimension**:
the same calendar date one/two years earlier, and a shift of 52/104 weeks.
Use historical BookingDate for the booking forecast and FlightDate for the flight
forecast. Report the search counts, including zero matches.

These are limited candidate linkage tests, not a proof of independence. Matching
markets, small ticket counts or approximate dates cannot establish common booking
identity without IDs or generation lineage. The forecasts use different event
dates. A zero-revenue booking margin need not connect to a zero-revenue flight
margin: a positive flight total can combine paying and zero-revenue allocations.


In [11]:
zero_inputs = {
    name: frame.loc[frame.Revenue.eq(0)].copy() for name, frame in frames.items()
}
zero_market_counts = (
    pd.concat(
        [
            group.groupby("FlightMarketCluster").size().rename(name)
            for name, group in zero_inputs.items()
        ],
        axis=1,
    )
    .fillna(0)
    .astype(int)
)
display(zero_market_counts)
zero_ticket_counts = (
    pd.concat(
        [
            group.groupby("Tix").size().rename(name)
            for name, group in zero_inputs.items()
        ],
        axis=1,
    )
    .fillna(0)
    .astype(int)
)
display(zero_ticket_counts)

zero_forecast_rows = pd.concat(
    [
        group.assign(dataset=name)
        for name, group in zero_inputs.items()
        if name != "actuals"
    ],
    ignore_index=True,
)
display(
    zero_forecast_rows[["dataset", "Date", "FlightMarketCluster", "Tix", "Revenue"]]
)

zero_link_checks = []
historical_zeros = zero_inputs["actuals"]
for name, history_date_column in [
    ("booking_forecast", "BookingDate"),
    ("flight_forecast", "FlightDate"),
]:
    targets = zero_inputs[name]
    for years in [1, 2]:
        for rule in ["same_calendar_date", "52_week_year"]:
            matched_targets = 0
            matched_historical_rows = 0
            for _, target in targets.iterrows():
                expected_date = (
                    target.Date - pd.DateOffset(years=years)
                    if rule == "same_calendar_date"
                    else target.Date - pd.Timedelta(weeks=52 * years)
                )
                hits = historical_zeros.loc[
                    historical_zeros.FlightMarketCluster.eq(target.FlightMarketCluster)
                    & historical_zeros[history_date_column].eq(expected_date)
                ]
                matched_targets += int(len(hits) > 0)
                matched_historical_rows += len(hits)
            zero_link_checks.append(
                {
                    "forecast": name,
                    "historical_date_basis": history_date_column,
                    "rule": rule,
                    "years_back": years,
                    "targets_checked": len(targets),
                    "matched_targets": matched_targets,
                    "matched_historical_rows": matched_historical_rows,
                }
            )
zero_link_summary = pd.DataFrame(zero_link_checks)
display(zero_link_summary)
print(
    "Market overlap is not proof of a shared underlying record. No tested alignment establishes lineage."
)

,actuals,booking_forecast,flight_forecast
FlightMarketCluster,,,
City,6,0,0
Greece,1,1,0
Iberian Peninsula,1,1,0
Mallorca,8,1,0
Balkans,0,0,1
Canaries,0,0,1


,actuals,booking_forecast,flight_forecast
Tix,,,
2.00,12,3,1
3.00,1,0,0
4.00,2,0,0
6.00,1,0,0
1.00,0,0,1


,dataset,Date,FlightMarketCluster,Tix,Revenue
0,booking_forecast,2026-02-04,Iberian Peninsula,2.00,0.00
1,booking_forecast,2026-02-11,Greece,2.00,0.00
2,booking_forecast,2026-04-02,Mallorca,2.00,0.00
3,flight_forecast,2026-05-29,Canaries,2.00,0.00
4,flight_forecast,2026-11-23,Balkans,1.00,0.00


,forecast,historical_date_basis,rule,years_back,targets_checked,matched_targets,matched_historical_rows
0,booking_forecast,BookingDate,same_calendar_date,1,3,0,0
1,booking_forecast,BookingDate,52_week_year,1,3,0,0
2,booking_forecast,BookingDate,same_calendar_date,2,3,0,0
3,booking_forecast,BookingDate,52_week_year,2,3,0,0
4,flight_forecast,FlightDate,same_calendar_date,1,2,0,0
5,flight_forecast,FlightDate,52_week_year,1,2,0,0
6,flight_forecast,FlightDate,same_calendar_date,2,2,0,0
7,flight_forecast,FlightDate,52_week_year,2,2,0,0


Market overlap is not proof of a shared underlying record. No tested alignment establishes lineage.


## 7. Which booking histories are incomplete?

For flights near the start of the file, bookings made earlier than 2024 may be
missing. For 2026 flights, bookings made after the end of 2025 are not observed.
An observed booking curve for such flights is **not final demand**.

We compare a fixed **last-365-days** window for flight dates whose entire window
falls inside the booking-date bounds. We start at the next complete month and
end at the observed final booking date. A 180-day view of those same flight dates
is a sensitivity check. Neither window proves that no bookings happened earlier.
This is our observation-window deduction, informed by P03, P07 and P13.

In [12]:
cross_year = a.groupby(
    [
        a.BookingDate.dt.year.rename("booking_year"),
        a.FlightDate.dt.year.rename("flight_year"),
    ]
).agg(tix=("Tix", "sum"), revenue_eur=("Revenue", "sum"))
display(cross_year)
window_start = OBSERVATION_START + pd.Timedelta(days=365)
COHORT_START = (
    window_start if window_start.day == 1 else window_start + pd.offsets.MonthBegin(1)
)
cohort_dates = timing.loc[timing.FlightDate.between(COHORT_START, OBSERVATION_END)]
cohort_365 = cohort_dates.loc[cohort_dates.lead_days.le(365)].copy()
cohort_180 = cohort_dates.loc[cohort_dates.lead_days.le(180)].copy()
assert COHORT_START - pd.Timedelta(days=365) >= OBSERVATION_START
assert cohort_365.FlightDate.max() <= OBSERVATION_END
print(
    "Common departure-date window:", COHORT_START.date(), "to", OBSERVATION_END.date()
)
sensitivity = pd.DataFrame(
    {
        "All recorded bookings": lead_summary(timing),
        "Same departures, last 365 booking days": lead_summary(cohort_365),
        "Same departures, last 180 booking days": lead_summary(cohort_180),
    }
).T
display(sensitivity)
print(
    "Different populations/windows are deliberate; differences are not an estimate of censoring bias."
)

month_pairs = (
    timing.assign(
        booking_month=timing.BookingDate.dt.to_period("M"),
        flight_month=timing.FlightDate.dt.to_period("M"),
    )
    .groupby(["booking_month", "flight_month"])
    .Tix.sum()
    .unstack(fill_value=0)
)
fig, ax = plt.subplots(figsize=(13, 6))
im = ax.imshow(np.log1p(month_pairs.to_numpy()), aspect="auto", cmap="Blues")
ax.set(
    xticks=np.arange(len(month_pairs.columns)),
    xticklabels=month_pairs.columns.astype(str),
    yticks=np.arange(len(month_pairs.index)),
    yticklabels=month_pairs.index.astype(str),
    xlabel="Flight month",
    ylabel="Booking month",
    title="Observed booking-to-flight months; blank cells mean no recorded tickets",
)
ax.tick_params(axis="x", labelrotation=90, labelsize=7)
ax.tick_params(axis="y", labelsize=8)
fig.colorbar(im, ax=ax, label="log(1 + observed travellers)")
fig.tight_layout()
fig.savefig(FIGURES / "booking_flight_months.png", dpi=150)
plt.close(fig)
display(
    Markdown("![Booking flight months](../reports/figures/booking_flight_months.png)")
)

tix   revenue_eur
booking_year flight_year                        
2024         2024        20,329.00 14,211,329.00
             2025         3,772.00  2,958,322.00
2025         2025        18,241.00 13,508,597.00
             2026         3,111.00  2,635,901.00

Common departure-date window: 2025-01-01 to 2025-12-31


,rows,tix,mean_days,p10_days,median_days,p90_days,within_30_days_pct,over_180_days_pct
All recorded bookings,"20,141.00","45,451.00",76.80,7.00,47.00,195.00,37.72,12.12
"Same departures, last 365 booking days","9,833.00","21,994.00",81.83,8.00,55.00,200.00,33.07,13.04
"Same departures, last 180 booking days","8,713.00","19,126.00",58.71,7.00,43.00,137.00,38.03,0.00


Different populations/windows are deliberate; differences are not an estimate of censoring bias.


![Booking flight months](../reports/figures/booking_flight_months.png)

## 8. Compare timing by market and season

The cumulative curve below answers: **by this many days before the flight,
what share of tickets recorded in the last 365 booking days had already been
booked?** The denominator is the last-365-day population, not all lifetime bookings.

Use the four largest eligible markets to keep the chart readable. Small markets
still appear in the table, with their evidence counts. [Research: P02, P03](../docs/literature_review.md)

In [13]:
cohort_market = pd.DataFrame(
    {
        name: lead_summary(group)
        for name, group in cohort_365.groupby("FlightMarketCluster")
    }
).T
cohort_market.index.name = "FlightMarketCluster"
display(cohort_market.sort_values("tix", ascending=False))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name in cohort_market.nlargest(4, "tix").index:
    group = cohort_365.loc[cohort_365.FlightMarketCluster.eq(name)]
    lead_mass = group.groupby("lead_days").Tix.sum().reindex(range(366), fill_value=0)
    booked_by = lead_mass.iloc[::-1].cumsum() / lead_mass.sum() * 100
    axes[0].plot(booked_by.index, booked_by.values, label=name)
axes[0].set(
    xlim=(365, 0),
    ylim=(0, 100),
    xlabel="Days remaining until departure",
    ylabel="Share booked by this point (%)",
    title="Observed cumulative curves within 365 days",
)
axes[0].legend(fontsize=8)
season_lead = pd.DataFrame(
    {
        int(month): lead_summary(group)
        for month, group in cohort_365.groupby(cohort_365.FlightDate.dt.month)
    }
).T
season_lead.index.name = "flight_month"
season_lead.median_days.plot(ax=axes[1], marker="o", color="#247BA0")
axes[1].set(
    xlabel="Flight month",
    ylabel="Traveller-weighted median lead time (days)",
    title="Timing by departure month in the eligible cohort",
    xticks=range(1, 13),
)
fig.tight_layout()
fig.savefig(FIGURES / "cohort_booking_patterns.png", dpi=150)
plt.close(fig)
display(
    Markdown(
        "![Cohort booking patterns](../reports/figures/cohort_booking_patterns.png)"
    )
)
display(season_lead[["rows", "tix", "median_days", "p90_days"]])

,rows,tix,mean_days,p10_days,median_days,p90_days,within_30_days_pct,over_180_days_pct
FlightMarketCluster,,,,,,,,
City,"5,068.00","11,242.00",75.96,8.00,49.00,190.00,35.88,11.15
Mallorca,"2,122.00","4,795.00",83.11,8.00,55.00,201.00,32.37,13.81
Greece,803.00,"1,871.00",100.38,9.00,74.00,235.00,27.26,19.29
Canaries,480.00,"1,004.00",85.31,10.00,59.00,200.00,29.18,13.05
Portugal Beach,277.00,601.00,94.86,14.00,72.00,220.00,22.30,14.64
Italy Beach,208.00,455.00,84.39,11.00,53.00,224.00,31.65,16.26
North Africa,174.00,411.00,73.24,10.00,57.00,157.00,27.01,6.33
Iberian Peninsula,159.00,362.00,107.30,10.00,83.00,252.00,24.86,21.82
Balearic Islands,125.00,268.00,90.56,8.00,61.00,219.00,33.58,17.54


![Cohort booking patterns](../reports/figures/cohort_booking_patterns.png)

,rows,tix,median_days,p90_days
flight_month,,,,
1,379.00,790.00,29.00,112.00
2,679.00,"1,436.00",33.00,118.00
3,975.00,"2,109.00",44.00,141.00
4,"1,083.00","2,547.00",63.00,163.00
5,"1,158.00","2,561.00",81.00,198.00
6,848.00,"1,916.00",72.00,188.00
7,778.00,"1,858.00",54.00,201.00
8,848.00,"2,018.00",47.00,212.00
9,"1,016.00","2,199.00",65.00,239.00


## 9. Revenue must be connected to ticket volume

AOV is revenue per traveller under the case-study definition. Group AOV is
`SUM(Revenue) / SUM(Tix)`, so a large row gets the appropriate ticket weight.
These summaries retain zero revenue and do not assume it is an error.
Differences by lead time or market are descriptive, not a causal effect of
booking earlier. The forecast views cover different populations, so their
AOVs need not be equal to one another. [Research: P03, P06, P08](../docs/literature_review.md)

In [14]:
LEAD_LABELS = ["0–7", "8–30", "31–90", "91–180", "181–365", "366+"]
timing["lead_band"] = pd.cut(
    timing.lead_days, [-1, 7, 30, 90, 180, 365, np.inf], labels=LEAD_LABELS
)
lead_aov = timing.groupby("lead_band", observed=False).agg(
    rows=("Tix", "size"), tix=("Tix", "sum"), revenue_eur=("Revenue", "sum")
)
lead_aov["aov_eur_per_traveller"] = lead_aov.revenue_eur / lead_aov.tix
lead_aov["ticket_share_pct"] = lead_aov.tix / timing.Tix.sum() * 100
display(lead_aov)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
lead_aov.ticket_share_pct.plot.bar(ax=axes[0], color="#247BA0", rot=0)
axes[0].set(
    title="When observed travellers booked",
    xlabel="Lead-time band (days)",
    ylabel="Traveller share (%)",
)
lead_aov.aov_eur_per_traveller.plot.bar(ax=axes[1], color="#D36B38", rot=0)
axes[1].set(
    title="Observed AOV by lead time",
    xlabel="Lead-time band (days)",
    ylabel="EUR per traveller",
)
fig.tight_layout()
fig.savefig(FIGURES / "aov_by_lead_time.png", dpi=150)
plt.close(fig)
display(Markdown("![Aov by lead time](../reports/figures/aov_by_lead_time.png)"))

weekday_rows = []
for year, group in a.groupby(a.BookingDate.dt.year):
    exposure = pd.Series(
        pd.date_range(group.BookingDate.min(), group.BookingDate.max()).dayofweek
    ).value_counts()
    for day, tickets in group.groupby(group.BookingDate.dt.dayofweek).Tix.sum().items():
        weekday_rows.append(
            {
                "booking_year": year,
                "weekday": ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"][day],
                "calendar_days": int(exposure[day]),
                "observed_tix_per_calendar_day": tickets / exposure[day],
            }
        )
print("Booking weekday comparison, accounting for the number of calendar days:")
display(
    pd.DataFrame(weekday_rows)
    .pivot(
        index="weekday", columns="booking_year", values="observed_tix_per_calendar_day"
    )
    .reindex(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
)

,rows,tix,revenue_eur,aov_eur_per_traveller,ticket_share_pct
lead_band,,,,,
0–7,2170,"4,588.00","2,818,389.00",614.30,10.09
8–30,5774,"12,554.00","8,116,646.00",646.54,27.62
31–90,6242,"13,765.00","10,278,506.00",746.71,30.29
91–180,3816,"9,036.00","7,348,561.00",813.25,19.88
181–365,2130,"5,485.00","4,727,127.00",861.83,12.07
366+,9,23.00,"24,570.00","1,068.26",0.05


![Aov by lead time](../reports/figures/aov_by_lead_time.png)

Booking weekday comparison, accounting for the number of calendar days:


booking_year,2024,2025
weekday,,
Mon,72.36,66.44
Tue,72.09,58.90
Wed,64.62,58.51
Thu,64.02,54.58
Fri,60.71,53.08
Sat,51.15,45.50
Sun,75.75,72.48


### 9a. Does the lead-time/AOV difference survive a similar market mix?

Use the same 2025 departure cohort and last-365-day window. Compare short lead
times (0–30 days) with long lead times (91–365); the middle band is outside this
specific diagnostic. Pair each market/flight-month stratum only when **both**
groups have at least five rows and ten travellers. These are descriptive sample
filters, not statistical guarantees. Report the excluded volume.

First compare ordinary group AOVs within those paired strata. Then use one
common set of market/month weights for both groups: each stratum's combined
short-plus-long tickets divided by all paired tickets. This holds the observed
market/month mix constant. It does not adjust for hotel quality, duration,
booking year, weekday or traveller characteristics and cannot establish that
booking earlier causes a higher price. Zero revenues and other review candidates
remain included. [Product-mix motivation: P14](../docs/literature_review.md#p14-henn-islam-schwind-and-wieland-2020)


In [15]:
MIX_MIN_ROWS = 5
MIX_MIN_TIX = 10
mix_cohort = cohort_365.assign(
    flight_month=cohort_365.FlightDate.dt.month,
    lead_group=np.where(
        cohort_365.lead_days.le(30),
        "short",
        np.where(cohort_365.lead_days.ge(91), "long", "middle"),
    ),
)
mix_selected = mix_cohort.loc[mix_cohort.lead_group.ne("middle")]
mix_groups = mix_selected.groupby(
    ["FlightMarketCluster", "flight_month", "lead_group"]
).agg(rows=("Tix", "size"), tix=("Tix", "sum"), revenue_eur=("Revenue", "sum"))
mix_pairs = mix_groups.unstack("lead_group").dropna()
mix_pairs = mix_pairs.loc[
    (mix_pairs["rows"] >= MIX_MIN_ROWS).all(axis=1)
    & (mix_pairs["tix"] >= MIX_MIN_TIX).all(axis=1)
]
if mix_pairs.empty:
    raise ValueError("No paired market/month strata meet the stated sample filters.")
mix_weights = mix_pairs["tix"].sum(axis=1) / mix_pairs["tix"].sum().sum()
assert np.isclose(mix_weights.sum(), 1)
mix_comparison_rows = []
for lead_group in ["short", "long"]:
    all_group = mix_selected.loc[mix_selected.lead_group.eq(lead_group)]
    paired_tix = mix_pairs["tix"][lead_group].sum()
    mix_comparison_rows.append(
        {
            "lead_group": lead_group,
            "all_selected_tix": all_group.Tix.sum(),
            "paired_tix": paired_tix,
            "paired_rows": int(mix_pairs["rows"][lead_group].sum()),
            "retained_ticket_share_pct": paired_tix / all_group.Tix.sum() * 100,
            "all_selected_aov_eur": all_group.Revenue.sum() / all_group.Tix.sum(),
            "paired_unadjusted_aov_eur": mix_pairs["revenue_eur"][lead_group].sum()
            / paired_tix,
            "common_mix_aov_eur": (
                mix_weights
                * mix_pairs["revenue_eur"][lead_group]
                / mix_pairs["tix"][lead_group]
            ).sum(),
        }
    )
mix_comparison = pd.DataFrame(mix_comparison_rows).set_index("lead_group")
display(mix_comparison)
print("Paired market/month strata:", len(mix_pairs))
print(
    "Tickets outside paired strata:",
    mix_selected.Tix.sum() - mix_pairs["tix"].sum().sum(),
)
mix_gaps = pd.Series(
    {
        "paired_unadjusted_gap_eur": mix_comparison.loc[
            "long", "paired_unadjusted_aov_eur"
        ]
        - mix_comparison.loc["short", "paired_unadjusted_aov_eur"],
        "common_mix_gap_eur": mix_comparison.loc["long", "common_mix_aov_eur"]
        - mix_comparison.loc["short", "common_mix_aov_eur"],
    }
)
display(mix_gaps.to_frame("long_minus_short_aov_eur"))
fig, ax = plt.subplots(figsize=(8, 4))
plot_mix = mix_comparison[["paired_unadjusted_aov_eur", "common_mix_aov_eur"]].T
plot_mix.index = ["Observed mix in paired strata", "Same market/month weights"]
plot_mix.rename(columns={"short": "0–30 days", "long": "91–365 days"}).plot.bar(
    ax=ax, color=["#247BA0", "#D36B38"], rot=0
)
ax.set(
    title="AOV and booking lead time: descriptive mix comparison",
    xlabel="",
    ylabel="EUR per traveller",
)
ax.legend(title="Lead time")
fig.tight_layout()
fig.savefig(FIGURES / "aov_mix_adjustment.png", dpi=150)
plt.close(fig)
display(Markdown("![AOV mix comparison](../reports/figures/aov_mix_adjustment.png)"))

,all_selected_tix,paired_tix,paired_rows,retained_ticket_share_pct,all_selected_aov_eur,paired_unadjusted_aov_eur,common_mix_aov_eur
lead_group,,,,,,,
short,"7,274.00","6,736.00",3218,92.60,646.61,620.04,644.37
long,"7,612.00","6,794.00",2789,89.25,844.58,805.59,779.71


Paired market/month strata: 61
Tickets outside paired strata: 1356.0


,long_minus_short_aov_eur
paired_unadjusted_gap_eur,185.55
common_mix_gap_eur,135.33


![AOV mix comparison](../reports/figures/aov_mix_adjustment.png)

## 10. Compare the supplied forecasts without changing them

The two annual totals describe different populations: bookings made in 2026 and
flights departing in 2026. Their difference is not automatically a mistake.
We need incoming bookings from prior years and outgoing flights in later years.

For each market and measure, if **B** is 2026 bookings, **F** is 2026 flights,
**I** is prior-year bookings for 2026 flights, and **O** is 2026 bookings for
later flights, then **O − I = B − F**. This identity needs a consistent scope.
A strictly 2026-by-2026 matrix cannot satisfy unequal enclosed marginal totals.
This follows from accounting and matrix-feasibility conditions (P08–P11).

In [16]:
booking_market = (
    b.groupby("FlightMarketCluster")[["Tix", "Revenue"]].sum().add_prefix("booking_")
)
flight_market = (
    f.groupby("FlightMarketCluster")[["Tix", "Revenue"]].sum().add_prefix("flight_")
)
market_gaps = booking_market.join(flight_market, how="outer")
market_gaps["tix_booking_minus_flight"] = (
    market_gaps.booking_Tix - market_gaps.flight_Tix
)
market_gaps["eur_booking_minus_flight"] = (
    market_gaps.booking_Revenue - market_gaps.flight_Revenue
)
market_gaps["booking_aov_eur"] = market_gaps.booking_Revenue / market_gaps.booking_Tix
market_gaps["flight_aov_eur"] = market_gaps.flight_Revenue / market_gaps.flight_Tix
display(market_gaps)
exact_tix_gap = (
    source_totals["booking_forecast"]["tix"] - source_totals["flight_forecast"]["tix"]
)
exact_eur_gap = (
    source_totals["booking_forecast"]["revenue_eur"]
    - source_totals["flight_forecast"]["revenue_eur"]
)
print("Exact booking-minus-flight gap:", exact_tix_gap, "tickets; EUR", exact_eur_gap)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
market_gaps.tix_booking_minus_flight.sort_values().plot.barh(
    ax=axes[0], color="#247BA0"
)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set(
    title="Booking minus flight forecast, by market", xlabel="Travellers", ylabel=""
)
monthly_fc = pd.DataFrame(
    {
        "Booking date": b.groupby(b.Date.dt.month).Tix.sum(),
        "Flight date": f.groupby(f.Date.dt.month).Tix.sum(),
    }
)
monthly_fc.plot(ax=axes[1], marker="o")
axes[1].set(
    title="Two forecast calendars (different populations)",
    xlabel="Month in 2026",
    ylabel="Forecast travellers",
    xticks=range(1, 13),
)
fig.tight_layout()
fig.savefig(FIGURES / "forecast_marginals.png", dpi=150)
plt.close(fig)
display(Markdown("![Forecast marginals](../reports/figures/forecast_marginals.png)"))

,booking_Tix,booking_Revenue,flight_Tix,flight_Revenue,tix_booking_minus_flight,eur_booking_minus_flight,booking_aov_eur,flight_aov_eur
FlightMarketCluster,,,,,,,,
Balearic Islands,284.31,"303,717.00",300.00,"316,692.00",-15.69,"-12,975.00","1,068.26","1,055.64"
Balkans,270.06,"247,398.00",279.00,"235,277.00",-8.94,"12,121.00",916.09,843.29
Canaries,"1,198.25","1,558,657.00",990.00,"1,274,546.00",208.25,"284,111.00","1,300.78","1,287.42"
City,"10,979.76","6,492,448.00","11,036.00","6,663,560.00",-56.24,"-171,112.00",591.31,603.80
Cyprus,112.61,"145,188.00",98.00,"126,970.00",14.61,"18,218.00","1,289.30","1,295.61"
Greece,"1,942.23","2,131,293.00","2,039.00","2,230,223.00",-96.77,"-98,930.00","1,097.34","1,093.78"
Iberian Peninsula,485.60,"490,397.00",415.00,"419,100.00",70.60,"71,297.00","1,009.88","1,009.88"
Intercont,198.02,"296,178.00",164.00,"247,663.00",34.02,"48,515.00","1,495.70","1,510.14"
Italy Beach,491.34,"488,540.00",527.00,"520,931.00",-35.66,"-32,391.00",994.30,988.48


Exact booking-minus-flight gap: 599.74 tickets; EUR 657734.00


![Forecast marginals](../reports/figures/forecast_marginals.png)

## 11. Can historical bookings for 2026 simply be held fixed?

Historical actuals include prior-year bookings for 2026 flights. These are
**observed on-books amounts**, not proof that the file contains every such
booking. Before treating them as fixed allocations, compare them with the flight
forecast at exactly the same flight date and market.

A negative remaining ticket or revenue requirement cannot be met by adding
nonnegative future amounts. Keys absent from the forecast are separately
reported as unspecified; they are not silently treated as zero forecasts.
Whether forecasts mean final totals or additional future sales needs clarification.

Keep all-market historical context separate from the illustrative annual
boundary balance. That balance uses only markets present in both forecasts;
Nordics has no supplied target and must not be mixed into their annual totals.
Daily targets that are absent within the shared markets are still unspecified.


In [17]:
on_books = a.loc[a.FlightDate.dt.year.eq(2026)].copy()
ob = (
    on_books.groupby(["FlightDate", "FlightMarketCluster"])[["Tix", "Revenue"]]
    .sum()
    .reset_index()
)
comparison = f.rename(columns={"Date": "FlightDate"}).merge(
    ob,
    on=["FlightDate", "FlightMarketCluster"],
    how="outer",
    suffixes=("_forecast", "_observed"),
    indicator=True,
)
shared = comparison.loc[comparison._merge.eq("both")].copy()
shared["remaining_tix"] = shared.Tix_forecast - shared.Tix_observed
shared["remaining_revenue_eur"] = shared.Revenue_forecast - shared.Revenue_observed
negative_tix = shared.loc[shared.remaining_tix.lt(0)]
negative_revenue = shared.loc[shared.remaining_revenue_eur.lt(0)]
unspecified = comparison.loc[comparison._merge.eq("right_only")]
on_books_summary = {
    "observed_rows_for_2026_flights": len(on_books),
    "observed_tix": float(on_books.Tix.sum()),
    "observed_revenue_eur": float(on_books.Revenue.sum()),
    "observed_keys_absent_from_flight_forecast": len(unspecified),
    "shared_keys_with_negative_ticket_residual": len(negative_tix),
    "shared_keys_with_negative_revenue_residual": len(negative_revenue),
}
display(pd.DataFrame([on_books_summary]).T.rename(columns={0: "value"}))
print("Conflicting shared keys (diagnostics, not corrected data):")
display(
    shared.loc[
        shared.remaining_tix.lt(0) | shared.remaining_revenue_eur.lt(0),
        [
            "FlightDate",
            "FlightMarketCluster",
            "Tix_forecast",
            "Tix_observed",
            "remaining_tix",
            "Revenue_forecast",
            "Revenue_observed",
            "remaining_revenue_eur",
        ],
    ]
)
print("Observed amounts at keys absent from the flight forecast:")
display(
    unspecified.groupby("FlightMarketCluster")[
        ["Tix_observed", "Revenue_observed"]
    ].sum()
)
common_forecast_markets = set(b.FlightMarketCluster) & set(f.FlightMarketCluster)
assert set(b.FlightMarketCluster) == set(f.FlightMarketCluster)
on_books_common = on_books.loc[
    on_books.FlightMarketCluster.isin(common_forecast_markets)
]
on_books_outside = on_books.loc[
    ~on_books.FlightMarketCluster.isin(common_forecast_markets)
]
annual_scope_rows = []
for label, population in [
    ("All historical markets: context only", on_books),
    ("Shared forecast markets: annual illustration", on_books_common),
    ("Outside forecast market scope: context only", on_books_outside),
]:
    annual_scope_rows.append(
        {
            "population": label,
            "rows": len(population),
            "tix": population.Tix.sum(),
            "revenue_eur": population.Revenue.sum(),
        }
    )
annual_scope = pd.DataFrame(annual_scope_rows)
display(annual_scope.set_index("population"))
# Preserve Decimal values for this accounting illustration.
raw_incoming = raw["actuals"].loc[
    pd.to_datetime(raw["actuals"].FlightDate).dt.year.eq(2026)
    & raw["actuals"].FlightMarketCluster.isin(common_forecast_markets)
]
annual_boundary_illustration = {
    "market_count": len(common_forecast_markets),
    "incoming_tix": sum(Decimal(str(value)) for value in raw_incoming.Tix),
    "incoming_revenue_eur": sum(Decimal(str(value)) for value in raw_incoming.Revenue),
}
annual_boundary_illustration["implied_outgoing_tix"] = (
    exact_tix_gap + annual_boundary_illustration["incoming_tix"]
)
annual_boundary_illustration["implied_outgoing_revenue_eur"] = (
    exact_eur_gap + annual_boundary_illustration["incoming_revenue_eur"]
)
print("Illustrative annual boundary balance for shared forecast markets:")
display(pd.DataFrame([annual_boundary_illustration]))
print(
    "Conditional on complete incoming bookings and forecast totals representing the complete annual scope. "
    "Absent daily targets remain unspecified. This does not resolve daily conflicts or prove feasibility."
)

,value
observed_rows_for_2026_flights,"1,327.00"
observed_tix,"3,111.00"
observed_revenue_eur,"2,635,901.00"
observed_keys_absent_from_flight_forecast,12.00
shared_keys_with_negative_ticket_residual,2.00
shared_keys_with_negative_revenue_residual,1.00


Conflicting shared keys (diagnostics, not corrected data):


,FlightDate,FlightMarketCluster,Tix_forecast,Tix_observed,remaining_tix,Revenue_forecast,Revenue_observed,remaining_revenue_eur
1400,2026-09-02,Mallorca,12.00,13.00,-1.00,"14,494.00","12,502.00","1,992.00"
1666,2026-10-10,Iberian Peninsula,3.00,9.00,-6.00,"5,576.00","7,355.00","-1,779.00"


Observed amounts at keys absent from the flight forecast:


,Tix_observed,Revenue_observed
FlightMarketCluster,,
Canaries,2.00,"3,464.00"
Iberian Peninsula,6.00,"6,310.00"
Nordics,13.00,"15,311.00"
North Africa,2.00,"3,038.00"
Portugal Beach,2.00,"2,142.00"
Turkey,6.00,"10,430.00"


,rows,tix,revenue_eur
population,,,
All historical markets: context only,1327,"3,111.00","2,635,901.00"
Shared forecast markets: annual illustration,1321,"3,098.00","2,620,590.00"
Outside forecast market scope: context only,6,13.00,"15,311.00"


Illustrative annual boundary balance for shared forecast markets:


,market_count,incoming_tix,incoming_revenue_eur,implied_outgoing_tix,implied_outgoing_revenue_eur
0,14,3098,2620590.00,3697.74,3278324.00


Conditional on complete incoming bookings and forecast totals representing the complete annual scope. Absent daily targets remain unspecified. This does not resolve daily conflicts or prove feasibility.


## 12. Sparsity, feasibility and an evaluation plan

**Sparse does not mean impossible.** Some markets have very few historical
records. Some potential date combinations are simply unobserved. Dates before
booking are a different kind of zero: normal chronology forbids those allocations.
A later model must preserve this distinction. [Research: P08–P11](../docs/literature_review.md)

**Accuracy and exact matching are different tests.** When a reconciliation
method is eventually fitted, measure marginal violations separately from errors
against held-out joint booking/flight observations. With supplied true margins,
a historical reconstruction is a conditional allocation test, not proof of the
accuracy of the original forecasts.

A future evaluation should simulate a booking-date cutoff, estimate patterns
only from information available by that cutoff, and restrict target populations
to an explicitly supported observation window. Use chronological validation,
simple baselines, ticket/euro errors, and distributional checks. Avoid MAPE
where actual denominators are zero. MASE also needs a valid nonzero reference
scale. We have explored both historical years here, so neither is an untouched
blind test set. [Research: P12, P13](../docs/literature_review.md)

**No model is fitted in this notebook, and no forecast accuracy score is reported.**

In [18]:
month_support = timing.assign(
    booking_month=timing.BookingDate.dt.month, flight_month=timing.FlightDate.dt.month
)
sparsity = month_support.groupby("FlightMarketCluster").agg(
    rows=("Tix", "size"), tix=("Tix", "sum")
)
sparsity["observed_month_pairs"] = (
    month_support[["FlightMarketCluster", "booking_month", "flight_month"]]
    .drop_duplicates()
    .groupby("FlightMarketCluster")
    .size()
)
sparsity["median_tix_per_record"] = month_support.groupby(
    "FlightMarketCluster"
).Tix.median()
display(sparsity.sort_values("rows"))
print(
    "Month-of-year pairs pool years: they describe observed seasonal support, not a dated feasibility mask."
)

metrics = {
    "booking_observation_start": str(OBSERVATION_START.date()),
    "booking_observation_end": str(OBSERVATION_END.date()),
    "source_totals": source_totals,
    "quality": quality.reset_index().to_dict(orient="records"),
    "review_thresholds": REVIEW_THRESHOLDS,
    "review_rule_summary": review_rules.to_dict(orient="records"),
    "review_dataset_summary": review_datasets.to_dict(orient="records"),
    "review_sensitivity": review_sensitivity.to_dict(orient="records"),
    "contextual_aov": contextual_aov.to_dict(orient="records"),
    "peer_min_rows": PEER_MIN_ROWS,
    "market_review_impact": market_review_impact.to_dict(orient="records"),
    "year_end_coverage": year_end_coverage.astype(object)
    .where(year_end_coverage.notna(), None)
    .to_dict(orient="records"),
    "forecast_targets_outside_illustrative_aov_bounds": len(
        forecast_aov_bound_conflicts
    ),
    "mix_comparison": mix_comparison.reset_index().to_dict(orient="records"),
    "mix_gaps": mix_gaps.to_dict(),
    "mix_paired_strata": len(mix_pairs),
    "mix_sample_filters": {
        "min_rows_per_group": MIX_MIN_ROWS,
        "min_tix_per_group": MIX_MIN_TIX,
    },
    "annual_scope": annual_scope.to_dict(orient="records"),
    "annual_boundary_illustration": annual_boundary_illustration,
    "monthly_scaling": monthly_scaling.astype(object)
    .where(monthly_scaling.notna(), None)
    .to_dict(orient="records"),
    "scaling_starting_factors": SCALING_STARTING_FACTORS,
    "scaling_coverage_rows": scaling_coverage,
    "zero_market_counts": zero_market_counts.reset_index().to_dict(orient="records"),
    "zero_ticket_counts": zero_ticket_counts.reset_index().to_dict(orient="records"),
    "zero_link_checks": zero_link_summary.to_dict(orient="records"),
    "calendar_support": calendar_rows,
    "negative_lead_rows": len(negative_lead),
    "negative_lead_tix": float(negative_lead.Tix.sum()),
    "negative_lead_revenue_eur": float(negative_lead.Revenue.sum()),
    "lead_all_observed": overall_lead,
    "lead_365_day_cohort": lead_summary(cohort_365),
    "cohort_departure_start": str(COHORT_START.date()),
    "cohort_departure_end": str(OBSERVATION_END.date()),
    "on_books": on_books_summary,
    "forecast_ticket_gap": str(exact_tix_gap),
    "forecast_revenue_gap_eur": str(exact_eur_gap),
    "market_ticket_share_pct": market.ticket_share_pct.to_dict(),
    "source_statement_ids": {
        name: frame.attrs.get("statement_id", "spark.sql")
        for name, frame in raw.items()
    },
}
(OUTPUTS / "metrics.json").write_text(json.dumps(metrics, indent=2, default=str) + "\n")
monthly_scaling.to_csv(OUTPUTS / "monthly_scaling_diagnostic.csv", index=False)
zero_market_counts.to_csv(OUTPUTS / "zero_revenue_market_counts.csv")
zero_link_summary.to_csv(OUTPUTS / "zero_revenue_link_checks.csv", index=False)
review_rules.to_csv(OUTPUTS / "anomaly_review_summary.csv", index=False)
review_sensitivity.to_csv(OUTPUTS / "anomaly_review_sensitivity.csv", index=False)
contextual_aov.to_csv(OUTPUTS / "contextual_aov_review.csv", index=False)
market_review_impact.to_csv(OUTPUTS / "market_review_impact.csv", index=False)
mix_comparison.to_csv(OUTPUTS / "aov_mix_comparison.csv")
market.to_csv(OUTPUTS / "market_summary.csv")
lead_market.to_csv(OUTPUTS / "lead_summary.csv")
market_gaps.to_csv(OUTPUTS / "forecast_market_gaps.csv")
shared.loc[shared.remaining_tix.lt(0) | shared.remaining_revenue_eur.lt(0)].to_csv(
    OUTPUTS / "on_books_conflicts.csv", index=False
)
print("Selected charts:", FIGURES.relative_to(ROOT))
print("Local generated summaries (Git-ignored):", OUTPUTS.relative_to(ROOT))

display(
    Markdown(f"""### Main observations from this run

- The exploratory review rules flag **{review_datasets.unique_flagged_rows.sum()}** distinct rows across the three inputs. Flags are not confirmed errors; review candidates remain in main summaries.
- The timing view excludes **{len(negative_lead)}** negative-lead row representing **{negative_lead.Tix.sum():,.0f} travellers**; raw tables are unchanged.
- Across recorded nonnegative lead times, the traveller-weighted median is **{overall_lead["median_days"]} days**, and the 90th percentile is **{overall_lead["p90_days"]} days**. These reflect the observed booking window.
- The 2026 booking forecast exceeds the flight forecast by **{exact_tix_gap:,.2f} travellers** and **EUR {exact_eur_gap:,.2f}**. Cross-year scope must be explicit.
- Prior-year observations include **{on_books.Tix.sum():,.0f} travellers** for 2026 flights. There are **{len(negative_tix)}** shared keys with negative ticket residuals, **{len(negative_revenue)}** with negative revenue residuals, and **{len(unspecified)}** observed keys absent from the flight forecast.
- **{len(forecast_aov_bound_conflicts)}** supplied forecast targets fall outside the exploratory AOV interval; that interval cannot be imposed on every joint cell.
- The near-one monthly scaling hypothesis can reconstruct **{scaling_coverage}** August–December rows; the inferred factors and integers are not verified original values.
- The tested calendar-year and 52-week zero-revenue alignments yield **{int(zero_link_summary.matched_targets.sum())}** matched targets; this does not prove independence.
- Before modelling, clarify final-versus-incremental forecast meaning, whether prior-year observations must stay fixed, and what absent forecast keys mean.

The evidence supports continuing with a transparent baseline and a feasibility specification. It does not yet select an algorithm or justify changing any supplied number.
""")
)

,rows,tix,observed_month_pairs,median_tix_per_record
FlightMarketCluster,,,,
Nordics,11,23.00,8,2.00
Cyprus,82,197.00,45,2.00
Malta,92,202.00,62,2.00
Intercont,179,413.00,81,2.00
Balkans,188,409.00,58,2.00
Turkey,272,731.00,80,2.00
Balearic Islands,279,591.00,70,2.00
North Africa,356,854.00,81,2.00
Italy Beach,365,780.00,79,2.00


Month-of-year pairs pool years: they describe observed seasonal support, not a dated feasibility mask.


Selected charts: reports/figures
Local generated summaries (Git-ignored): outputs/analysis


### Main observations from this run

- The exploratory review rules flag **47** distinct rows across the three inputs. Flags are not confirmed errors; review candidates remain in main summaries.
- The timing view excludes **1** negative-lead row representing **2 travellers**; raw tables are unchanged.
- Across recorded nonnegative lead times, the traveller-weighted median is **47 days**, and the 90th percentile is **195 days**. These reflect the observed booking window.
- The 2026 booking forecast exceeds the flight forecast by **599.74 travellers** and **EUR 657,734.00**. Cross-year scope must be explicit.
- Prior-year observations include **3,111 travellers** for 2026 flights. There are **2** shared keys with negative ticket residuals, **1** with negative revenue residuals, and **12** observed keys absent from the flight forecast.
- **7** supplied forecast targets fall outside the exploratory AOV interval; that interval cannot be imposed on every joint cell.
- The near-one monthly scaling hypothesis can reconstruct **1045** August–December rows; the inferred factors and integers are not verified original values.
- The tested calendar-year and 52-week zero-revenue alignments yield **0** matched targets; this does not prove independence.
- Before modelling, clarify final-versus-incremental forecast meaning, whether prior-year observations must stay fixed, and what absent forecast keys mean.

The evidence supports continuing with a transparent baseline and a feasibility specification. It does not yet select an algorithm or justify changing any supplied number.


## 13. EDA conclusions and hand-off to modelling

This completes the initial exploratory analysis. Root causes of individual
dummy anomalies remain unknown; the scope and feasibility decisions below must
be explicit before fitting an allocation model. No model, deployment or forecast
accuracy result is produced by this notebook.

| Finding | Current treatment | Consequence for the next stage |
| --- | --- | --- |
| One negative lead-time row | Retain raw data; exclude from timing analysis | Forbid flights before booking in valid joint allocations |
| Zero revenue with positive Tix | Retain supplied values; compare markets and limited date links | Preserve zero-revenue margins; do not invent a refund/free-product cause |
| Fractional Tix and monthly scaling pattern | Preserve exact targets; document compatible factors as a hypothesis | Use continuous expected volumes; inferred integers are not corrected targets |
| Extreme AOVs | Flag with peer context and sensitivity; retain | Universal review cutoffs conflict with seven supplied target margins |
| Nordics and absent date/market targets | Retain historical context; treat absent targets as unspecified | Define output scope, including 31 December, explicitly |
| Cross-year flows | State accounting identity on the same market scope | Include supported dates outside the forecast year; define boundary assumptions |
| Observed amounts exceed two daily targets | Preserve and report the conditional conflict | Decide whether on-books amounts are fixed and forecasts are final or incremental |
| Finite observation window and sparse markets | Use covered lookback cohorts and report sample sizes | Choose appropriate pooling and temporal validation; do not assume unobserved cells impossible |
| Lead-group AOV changes under common market/month weights | Report descriptive association and missing product fields | Validate ticket/revenue allocations jointly; do not claim a causal price effect |

**Next deliverable:** an explicit modelling specification and transparent
baseline that links BookingDate and FlightDate per market, preserves the agreed
daily Tix/Revenue margins and respects chronology/nonnegativity. Evaluate
constraint violations separately from historical allocation quality. Any
experiment using inferred integer counts must be labelled a sensitivity scenario
and kept separate from the solution matching the original forecasts.
